# 1. Data Preprocessing, Normalizing: Tiền xử lí dữ liệu, EDA, Chuẩn hóa dữ liệu.

In [1]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.stem.porter import PorterStemmer

# ----------------------------------------------------------
# BƯỚC 1: KHỞI TẠO CÔNG CỤ NLP VÀ ĐỊNH NGHĨA HÀM
# ----------------------------------------------------------
nltk.download('punkt') 
nltk.download('punkt_tab')
stemmer = PorterStemmer()

def tokenize(sentence):
    """Tách câu thành các từ riêng lẻ."""
    return nltk.word_tokenize(sentence)

def stem(word):
    """Đưa từ về dạng gốc và viết thường."""
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, words):
    """Tạo mảng 0 và 1 đánh dấu sự xuất hiện của từ."""
    sentence_words = [stem(word) for word in tokenized_sentence]
    bag = np.zeros(len(words), dtype=np.float32)
    for idx, w in enumerate(words):
        if w in sentence_words: 
            bag[idx] = 1
    return bag

# ----------------------------------------------------------
# BƯỚC 2: ĐỌC VÀ GỘP DỮ LIỆU (DATA INTEGRATION)
# ----------------------------------------------------------
# 2.1. Đọc dữ liệu JSON
with open('intents.json', 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

# 2.2. Đọc dữ liệu CSV
df = pd.read_csv('Mental_Health_FAQ.csv')

# 2.3. Gộp CSV vào JSON
for index, row in df.iterrows():
    question_id = str(row['Question_ID'])
    question = str(row['Questions'])
    answer = str(row['Answers'])
    
    new_intent = {
        "tag": f"faq_{question_id}", 
        "patterns": [question],      
        "responses": [answer]        
    }
    intents_data['intents'].append(new_intent)

print(f"Đã gộp thành công! Tổng số ý định (intents) hiện tại là: {len(intents_data['intents'])}")

# ----------------------------------------------------------
# BƯỚC 3: TIỀN XỬ LÝ (NLP PREPROCESSING)
# ----------------------------------------------------------
all_words = [] 
tags = []      
xy = []        

# Duyệt qua toàn bộ dữ liệu ĐÃ GỘP
for intent in intents_data['intents']:
    tag = intent['tag']
    tags.append(tag)
    
    for pattern in intent['patterns']:
        w = tokenize(pattern) 
        all_words.extend(w)
        xy.append((w, tag))

# Làm sạch từ vựng (bỏ dấu câu, tìm từ gốc, xóa trùng lặp)
ignore_words = ['?', '!', '.', ',']
all_words = [stem(w) for w in all_words if w not in ignore_words]
all_words = sorted(set(all_words))
tags = sorted(set(tags))

print(f"Tổng số từ vựng (Vocabulary size): {len(all_words)}")
print(f"Tổng số nhãn (Classes): {len(tags)}")

# ----------------------------------------------------------
# BƯỚC 4: TẠO DỮ LIỆU HUẤN LUYỆN (X_train và y_train)
# ----------------------------------------------------------
X_train = []
y_train = []

for (pattern_sentence, tag) in xy:
    # Biến câu thành mảng số Bag of Words
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    
    # Lấy index của nhãn dự đoán
    label = tags.index(tag)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Kích thước X_train (Dữ liệu đầu vào):", X_train.shape)
print("Kích thước y_train (Nhãn dự đoán):", y_train.shape)

with open('merged_intents.json', 'w', encoding='utf-8') as f:
    json.dump(intents_data, f, ensure_ascii=False, indent=4)

print("Đã xuất file thành công: merged_intents.json")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hovin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hovin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Đã gộp thành công! Tổng số ý định (intents) hiện tại là: 178
Tổng số từ vựng (Vocabulary size): 411
Tổng số nhãn (Classes): 178
Kích thước X_train (Dữ liệu đầu vào): (330, 411)
Kích thước y_train (Nhãn dự đoán): (330,)
Đã xuất file thành công: merged_intents.json


# 2. Model Architecture: Xây dựng mô hình.

In [2]:
import torch
import torch.nn as nn

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        
        # Lớp tuyến tính 1: Nhận Input (411) -> Truyền vào tầng ẩn thứ nhất (hidden_size)
        self.l1 = nn.Linear(input_size, hidden_size) 
        
        # Lớp tuyến tính 2: Tầng ẩn thứ nhất -> Tầng ẩn thứ hai
        self.l2 = nn.Linear(hidden_size, hidden_size) 
        
        # Lớp đầu ra: Tầng ẩn thứ hai -> Đầu ra Output (178 classes)
        self.l3 = nn.Linear(hidden_size, num_classes)
        
        # Hàm kích hoạt ReLU (Rectified Linear Unit) để tạo tính phi tuyến cho mô hình
        self.relu = nn.ReLU()
    
    def forward(self, x):
        """
        Hàm forward định nghĩa con đường mà dữ liệu X_train sẽ chảy qua mạng nơ-ron.
        """
        # Dữ liệu đi qua lớp 1 và bị "bẻ cong" bởi hàm kích hoạt ReLU
        out = self.l1(x)
        out = self.relu(out)
        
        # Tiếp tục đi qua lớp 2 và lại qua ReLU
        out = self.l2(out)
        out = self.relu(out)
        
        # Đi qua lớp cuối cùng để xuất ra kết quả (Logits)
        out = self.l3(out)
        
        # LƯU Ý QUAN TRỌNG: Không dùng hàm Softmax ở lớp cuối cùng!
        return out

# 3. Model Training: Train Mô hình Chatbot.

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. CÀI ĐẶT SIÊU THAM SỐ (Bảng điều khiển)
# ==========================================
num_epochs = 1000        # Số lần học lặp lại toàn bộ dữ liệu
batch_size = 8           # Số câu hỏi học trong một mẻ
learning_rate = 0.001    # Tốc độ học (Bước chân của Gradient Descent)

input_size = len(X_train[0])
hidden_size = 8
output_size = len(tags)

# ==========================================
# 2. ĐÓNG GÓI DỮ LIỆU CHO PYTORCH
# ==========================================
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        self.x_data = X_train
        self.y_data = y_train

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

dataset = ChatDataset()
# DataLoader giúp tự động chia nhỏ dữ liệu thành các batch (mẻ) và xáo trộn (shuffle)
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

# ==========================================
# 3. KHỞI TẠO MÔ HÌNH, TÍNH LOSS & TỐI ƯU HÓA
# ==========================================
# Kiểm tra xem máy có Card đồ họa (GPU) không, nếu không thì dùng CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Gọi bản vẽ NeuralNet ra để tạo thành mô hình thực tế
model = NeuralNet(input_size, hidden_size, output_size).to(device)

# Hàm tính sai số (CrossEntropyLoss đã bao gồm sẵn Softmax)
criterion = nn.CrossEntropyLoss()

# Thuật toán tối ưu trọng số dựa trên đạo hàm (Nâng cấp từ Gradient Descent)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# ==========================================
# 4. VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP)
# ==========================================
print("Bắt đầu huấn luyện mô hình...")

for epoch in range(num_epochs):
    for (words, labels) in train_loader:
        words = words.to(device)
        labels = labels.to(dtype=torch.long).to(device)
        
        # Bước 1: Forward pass (Đưa dữ liệu qua mạng để dự đoán)
        outputs = model(words)
        
        # Bước 2: Tính sai số giữa dự đoán và đáp án thực tế
        loss = criterion(outputs, labels)
        
        # Bước 3: Backward pass (Lan truyền ngược để tính đạo hàm) & Cập nhật trọng số
        optimizer.zero_grad() # Xóa rác đạo hàm cũ
        loss.backward()       # Tính đạo hàm mới
        optimizer.step()      # Bước xuống dốc (cập nhật trọng số)
        
    # Cứ sau 100 epochs, in kết quả ra màn hình cho dễ theo dõi
    if (epoch+1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss (Sai số): {loss.item():.4f}')

print(f'Hoàn tất! Sai số cuối cùng: {loss.item():.4f}')

# ==========================================
# 5. LƯU LẠI "BỘ NÃO" ĐỂ DÙNG SAU
# ==========================================
data = {
    "model_state": model.state_dict(), # Lưu các trọng số đã được tối ưu
    "input_size": input_size,
    "hidden_size": hidden_size,
    "output_size": output_size,
    "all_words": all_words,
    "tags": tags
}

FILE = "chatbot_model.pth"
torch.save(data, FILE)
print(f'Đã lưu file mô hình tại: {FILE}')

Bắt đầu huấn luyện mô hình...
Epoch [100/1000], Loss (Sai số): 0.1007
Epoch [200/1000], Loss (Sai số): 0.0169
Epoch [300/1000], Loss (Sai số): 0.8627
Epoch [400/1000], Loss (Sai số): 0.0002
Epoch [500/1000], Loss (Sai số): 0.0000
Epoch [600/1000], Loss (Sai số): 0.0000
Epoch [700/1000], Loss (Sai số): 0.0000
Epoch [800/1000], Loss (Sai số): 0.0000
Epoch [900/1000], Loss (Sai số): 0.4591
Epoch [1000/1000], Loss (Sai số): 0.3880
Hoàn tất! Sai số cuối cùng: 0.3880
Đã lưu file mô hình tại: chatbot_model.pth


# 4. Infering & Chatting: Tạo luồng - Chạy thử bot.

In [1]:
import json
import torch
import torch.nn as nn
import numpy as np
import random
import nltk
from nltk.stem.porter import PorterStemmer

# ==========================================
# 1. KHAI BÁO LẠI CÔNG CỤ NLP VÀ KIẾN TRÚC MÔ HÌNH
# ==========================================
stemmer = PorterStemmer()

def tokenize(sentence):
    return nltk.word_tokenize(sentence)

def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, words):
    sentence_words = [stem(word) for word in tokenized_sentence]
    bag = np.zeros(len(words), dtype=np.float32)
    for idx, w in enumerate(words):
        if w in sentence_words: 
            bag[idx] = 1
    return bag

# Bản thiết kế "ngôi nhà" (Bắt buộc phải có để đắp trọng số vào)
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size) 
        self.l2 = nn.Linear(hidden_size, hidden_size) 
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

# ==========================================
# 2. TẢI DỮ LIỆU GỘP VÀ "BỘ NÃO" ĐÃ HUẤN LUYỆN
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ĐỌC FILE JSON MỚI ĐÃ GỘP (Rất quan trọng!)
with open('merged_intents.json', 'r', encoding='utf-8') as f:
    intents_data = json.load(f)

# Tải "nội thất" (trọng số) từ file pth
FILE = "chatbot_model.pth"
data = torch.load(FILE)

input_size = data["input_size"]
hidden_size = data["hidden_size"]
output_size = data["output_size"]
all_words = data['all_words']
tags = data['tags']
model_state = data["model_state"]

# Dựng khung nhà và đắp nội thất vào
model = NeuralNet(input_size, hidden_size, output_size).to(device)
model.load_state_dict(model_state)
model.eval() # Bật chế độ suy luận (không học thêm nữa)

# ==========================================
# 3. VÒNG LẶP CHAT TƯƠNG TÁC
# ==========================================
bot_name = "Meiritsuu"
print("Chatbot đã sẵn sàng! Gõ 'quit' để thoát.")
print("-" * 50)

while True:
    sentence = input("Bạn: ")
    if sentence.lower() == "quit":
        print(f"{bot_name}: Tạm biệt! Chúc bạn một ngày tốt lành.")
        break

    # Băm câu chữ thành số
    sentence_tokenized = tokenize(sentence)
    X = bag_of_words(sentence_tokenized, all_words)
    X = X.reshape(1, X.shape[0])
    X = torch.from_numpy(X).to(device)

    # Đưa cho AI dự đoán
    with torch.no_grad():
        output = model(X)
    
    _, predicted = torch.max(output, dim=1)
    tag_name = tags[predicted.item()] # Nhãn mà bot đoán ra

    # Tính tỷ lệ tự tin (%)
    probs = torch.softmax(output, dim=1)
    prob = probs[0][predicted.item()]

    # Nếu tự tin trên 75% thì mới trả lời
    if prob.item() > 0.75:
        # Cơ chế an toàn
        if tag_name == "suicide":
            print(f"{bot_name}: TÔI RẤT LO LẮNG CHO BẠN. Xin hãy gọi ngay đến Đường dây nóng Hỗ trợ Tâm lý Quốc gia 111 hoặc 115. Bạn không cô đơn đâu!")
            continue 
        
        # Lục tìm câu trả lời trong mảng dữ liệu
        for intent in intents_data['intents']:
            if tag_name == intent["tag"]: 
                print(f"{bot_name}: {random.choice(intent['responses'])}")
                break
    else:
        print(f"{bot_name}: Xin lỗi, tôi không chắc mình hiểu đúng ý bạn. Bạn có thể nói rõ hơn được không?")

ModuleNotFoundError: No module named 'nltk'